In [ ]:
import os

# Set to the root of the merfish_pancancer dataset directory,
# or export MERFISH_DATA_DIR before running the notebook.
BASE_DATA_DIR = os.environ.get("MERFISH_DATA_DIR", "/path/to/merfish_pancancer")
ATLAS_DIR = os.environ.get("ATLAS_DIR", "/path/to/atlases")

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import plotnine as gg
import glob
from spatial_correction.datanavigation import MerfishDataNavigator


def get_reference_lfcs(adata, cell_types):
    x1 = adata[adata.obs["cell_type"].isin(cell_types), :].X.toarray()
    x2 = adata[~adata.obs["cell_type"].isin(cell_types), :].X.toarray()
    x1_mean = x1.mean(axis=0)
    x2_mean = x2.mean(axis=0)

    pct_exp = (x1 > 1e-6).mean(0)
    mean_exp = x1.mean(0)
    return pd.DataFrame(
        {
            "lfc": x1_mean - x2_mean,
            "pct_exp": pct_exp,
            "mean_exp": mean_exp,
            "gene_id": adata.var_names,
            "feature_name": adata.var["feature_name"],
        }
    )

In [ ]:
t_cell_types = [
    "effector memory CD4-positive, alpha-beta T cell",
    "effector memory CD8-positive, alpha-beta T cell",
    "regulatory T cell",
    "mucosal invariant T cell",
    "CD4-positive, alpha-beta T cell",
    "effector memory CD8-positive, alpha-beta T cell, terminally differentiated",
    "central memory CD4-positive, alpha-beta T cell",
    "T cell",
    "gamma-delta T cell",
]

fibroblast_types = [
    "alveolar type 1 fibroblast cell",
    "fibroblast",
    "fibroblast of lung",
    "mesothelial cell",
    "muscle cell",
    "myofibroblast cell",
    "smooth muscle cell of the pulmonary artery",
    "pericyte",
    "stromal cell",
    "endothelial cell of periportal hepatic sinusoid",
    "endothelial cell of pericentral hepatic sinusoid",
]

t_cell_markers = [
    ("CD3E", "blue"),
    ("CD3D", "blue"),
    ("CD3G", "blue"),
    ("CD4", "blue"),
    ("CD8A", "blue"),
    ("CD8B", "blue"),
]

fibroblast_markers = [
    ("COL1A1", "blue"),
    ("FAP", "blue"),
    ("PDFGRA", "blue"),
]

markers_for_other_cells = [
    ("CD3E", "green"),
    ("CD3D", "green"),
    ("CD3G", "green"),
    ("CD4", "green"),
    ("CD8A", "green"),
    ("CD8B", "green"),
    ("COL1A1", "red"),
    ("FAP", "red"),
    ("PDFGRA", "red"),
]

In [ ]:
lung1_adata_path = f"{BASE_DATA_DIR}/lung1_adata.annotated.h5ad"
lung2_adata_path = f"{BASE_DATA_DIR}/lung2_adata.annotated.h5ad"
colon1_adata_path = f"{BASE_DATA_DIR}/colon1_adata.annotated.h5ad"
colon2_adata_path = f"{BASE_DATA_DIR}/colon2_adata.annotated.h5ad"


adata_lung1 = sc.read_h5ad(lung1_adata_path)
adata_lung2 = sc.read_h5ad(lung2_adata_path)
adata_colon1 = sc.read_h5ad(colon1_adata_path)
adata_colon2 = sc.read_h5ad(colon2_adata_path)

In [ ]:
dataname_to_assay_path = {
    "lung1": f"{BASE_DATA_DIR}/HumanLungCancerPatient1",
    "lung2": f"{BASE_DATA_DIR}/HumanLungCancerPatient2",
    "colon1": f"{BASE_DATA_DIR}/HumanColonCancerPatient1",
    "colon2": f"{BASE_DATA_DIR}/HumanColonCancerPatient2",
}

# Extracting reference LFCs

## Pan-Cancer

In [ ]:
adata = sc.read_h5ad(f"{ATLAS_DIR}/Guimaraes_2024/Guimaraes_2024.h5ad")
adata

In [ ]:
def compare_exp(adata):
    x1 = adata[adata.obs["harm_sample.type"] == "tumor", :].X.toarray()
    x2 = adata[adata.obs["harm_sample.type"] == "normal", :].X.toarray()
    x1_mean = x1.mean(axis=0)
    x2_mean = x2.mean(axis=0)

    pct_exp = (x1 > 1e-6).mean(0)
    mean_exp = x1.mean(0)

    feature_names = adata.var["feature_name"].str.split("_").str[0].values
    return pd.DataFrame(
        {
            "lfc": x1_mean - x2_mean,
            "pct_exp": pct_exp,
            "mean_exp": mean_exp,
            "gene_id": adata.var_names,
            "feature_name": feature_names,
        }
    )

In [ ]:
adata_t_cell = adata[adata.obs["cell_type"] == "T cell"].copy()
adata_lung = adata_t_cell[adata_t_cell.obs["tissue"] == "lung"].copy()
adata_colon = adata_t_cell[adata_t_cell.obs["tissue"] == "colorectum"].copy()

res_lung = compare_exp(adata_lung).assign(dataset="lung", cell_type="t cell")
res_colon = compare_exp(adata_colon).assign(dataset="colon", cell_type="t cell")

In [ ]:
res = pd.concat([res_lung, res_colon], axis=0)
res.to_csv("results/all_lfcs_Guimaraes_tcell.tsv", index=False, sep="\t")

In [ ]:
adata_cd8t_cell = adata[adata.obs["author_cell_type"].str.startswith("TCD8")].copy()
adata_lung_cd8t = adata_cd8t_cell[adata_cd8t_cell.obs["tissue"] == "lung"].copy()
adata_colon_cd8t = adata_cd8t_cell[adata_cd8t_cell.obs["tissue"] == "colorectum"].copy()

res_lung_cd8t = compare_exp(adata_lung_cd8t).assign(dataset="lung", cell_type="t cell")
res_colon_cd8t = compare_exp(adata_colon_cd8t).assign(dataset="colon", cell_type="t cell")

In [17]:
res_cd8t = pd.concat([res_lung_cd8t, res_colon_cd8t], axis=0)
res_cd8t.to_csv("results/all_lfcs_Guimaraes_cd8t.tsv", index=False, sep="\t")

## HCA

In [ ]:
ref_lung = sc.read_h5ad(f"{ATLAS_DIR}/HCA/HCA_lung.h5ad")
ref_colon = sc.read_h5ad(
    f"{ATLAS_DIR}/HCA/HCA_intestine.h5ad"
)

In [ ]:
datasets = [
    ("lung", ref_lung),
    ("colon", ref_colon),
]

cell_types = [
    ("t cell", t_cell_types),
    ("fibroblast", fibroblast_types),
]

all_lfcs = []
for dataset_name, ref_adata in datasets:
    for cell_type_name, cell_type_li in cell_types:
        lfcs = get_reference_lfcs(ref_adata, cell_type_li)
        lfcs["dataset"] = dataset_name
        lfcs["cell_type"] = cell_type_name
        all_lfcs.append(lfcs)
all_lfcs = pd.concat(all_lfcs)
all_lfcs.to_csv("all_lfcs.tsv", index=False, sep="\t")

# Data export


In [ ]:
def weight_obs(adata, w_t_cell=10, w_fibroblast=10):
    p = np.ones(adata.shape[0])
    p[adata.obs["cell_type"] == "t cell"] = w_t_cell
    p[adata.obs["cell_type"] == "fibroblast"] = w_fibroblast
    tilde_p = p / p.sum()
    return tilde_p, p


def get_subsample(adata, n_cells, w_t_cell, w_fibroblast):
    tilde_p, p = weight_obs(adata, w_t_cell, w_fibroblast)
    adata.obs["sampling_weight"] = p
    random_indices = np.random.choice(
        range(adata.shape[0]), size=n_cells, p=tilde_p, replace=False
    )
    return adata[random_indices, :]

In [ ]:
adata_lung2.obs["sampling_weight"].value_counts()

In [ ]:
adata_colon1.obs["sampling_weight"].value_counts()

In [ ]:
adata_colon2.obs["sampling_weight"].value_counts()

In [ ]:
adata_colon1.obs["cell_type"].value_counts()

In [ ]:
np.random.seed(0)
adata_lung1_sub = get_subsample(adata_lung1, 600, w_t_cell=10, w_fibroblast=10)
adata_lung2_sub = get_subsample(adata_lung2, 600, w_t_cell=10, w_fibroblast=10)
adata_colon1_sub = get_subsample(adata_colon1, 600, w_t_cell=80, w_fibroblast=5)
adata_colon2_sub = get_subsample(adata_colon2, 600, w_t_cell=15, w_fibroblast=5)

In [ ]:
# saving importance weights
adata_lung1.write_h5ad(lung1_adata_path)
adata_lung2.write_h5ad(lung2_adata_path)
adata_colon1.write_h5ad(colon1_adata_path)
adata_colon2.write_h5ad(colon2_adata_path)

In [ ]:
(
    gg.ggplot(all_lfcs.query('dataset == "colon"'))
    + gg.aes(x="lfc")
    + gg.geom_histogram(bins=100)
    + gg.theme_minimal()
    + gg.facet_wrap("~cell_type", scales="free")
)

In [ ]:
adata_sub = adata_colon1_sub
adata_all = adata_colon1
path_to_assay_data = dataname_to_assay_path["colon1"]
lfcs = all_lfcs.query("dataset == 'colon'")

In [ ]:
def export_cell_info(adata_sub, adata_all, path_to_assay_data, lfcs):
    path_to_images = glob.glob(os.path.join(path_to_assay_data, "images/*.tif"))
    boundary_geojson_path = os.path.join(
        path_to_assay_data, "proseg_results/cell-polygons.geojson"
    )
    all_idx_sampled = adata_sub.obs.index
    export_dir = os.path.join(path_to_assay_data, "manual_annotations/images")
    os.makedirs(export_dir, exist_ok=True)

    navigator = MerfishDataNavigator(
        path_to_assay_data,
        adata_all,
        path_to_images,
        boundary_geojson_path=boundary_geojson_path,
    )

    for idx in tqdm(all_idx_sampled):
        ct = adata_sub.obs.loc[idx, "cell_type"]
        if ct == "t cell":
            de_res_markers = lfcs.query("cell_type == 't cell'").set_index(
                "feature_name"
            )
            gene_color_pairs = t_cell_markers
        elif ct == "fibroblast":
            de_res_markers = lfcs.query("cell_type == 'fibroblast'").set_index(
                "feature_name"
            )
            gene_color_pairs = fibroblast_markers
        else:
            de_res_markers = lfcs.query("cell_type == 't cell'").set_index(
                "feature_name"
            )
            gene_color_pairs = markers_for_other_cells
        fig = navigator.plot_cell_info(
            cell_idx=idx,
            de_res_markers=de_res_markers,
            gene_color_pairs=gene_color_pairs,
            suptitle=ct,
            normalize_range=1.0,
            delta=25,
        )
        fig.savefig(os.path.join(export_dir, f"cell_{idx}.png"), dpi=300)

In [ ]:
export_cell_info(
    adata_sub=adata_colon1_sub,
    adata_all=adata_colon1,
    path_to_assay_data=dataname_to_assay_path["colon1"],
    lfcs=all_lfcs.query("dataset == 'colon'"),
)

In [ ]:
export_cell_info(
    adata_sub=adata_colon2_sub,
    adata_all=adata_colon2,
    path_to_assay_data=dataname_to_assay_path["colon2"],
    lfcs=all_lfcs.query("dataset == 'colon'"),
)